In [16]:
import ast
import json
from copy import deepcopy

import pandas as pd
import requests
from tqdm import tqdm

API_URL = "http://localhost:8080/v1/chat/completions"

SYSTEM_PROMPT_TEMPLATE = """Ты — лемматизатор географических названий.

Твоя задача — для каждого слова вернуть нормализованную географическую форму в именительном падеже.
Если слово является прилагательным, обозначающим страну или город (например, "Московский", "Французской", "Туркменский") — верни существительное, которое оно обозначает ("Москва", "Франция", "Туркмения").
Если слово не удаётся однозначно интерпретировать, просто приведи его к нормальной форме.
Никаких пояснений, текста или комментариев — только JSON.

Формат ответа:
{
  "оригинал": "нормализованная_форма",
  ...
}

Теперь обработай:
%s
"""

PROMPT_TEMPLATE = {
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT_TEMPLATE},
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": "%s"},
    ],
    "max_tokens": 800,
    "temperature": 0.1,
    "top_p": 0.9
}


def get_prompt(batch):
    """Создает корректный промпт для списка строк"""
    text = json.dumps(batch, ensure_ascii=False)
    prompt = deepcopy(PROMPT_TEMPLATE)
    prompt["messages"][0]["content"] = prompt["messages"][0]["content"] % text
    prompt["messages"][2]["content"] = text
    return prompt


def classify_geo_entities_llm(batch, api_url=API_URL):
    """Отправляет список названий в LLM и возвращает словарь {оригинал: нормализованная форма}"""
    prompt = get_prompt(batch)

    try:
        response = requests.post(api_url, json=prompt, timeout=90)
        response.raise_for_status()
        content = response.json()["choices"][0]["message"]["content"]
    except Exception as e:
        print(f"Ошибка запроса: {e}")
        return {}

    try:
        parsed = json.loads(content)
        if isinstance(parsed, dict):
            return parsed
        elif isinstance(parsed, list):
            result = {}
            for item in parsed:
                if isinstance(item, dict):
                    result.update(item)
            return result
    except json.JSONDecodeError:
        print(f"⚠️ Не удалось распарсить ответ:\n{content}")
        return {}

    return {}


def normalize_loc_list(entities):
    """Нормализует один список (одну строку df['loc']) через LLM"""
    tqdm.pandas(desc="Нормализация loc через LLM")
    if not entities:
        return []

    if not isinstance(entities, list):
        try:
            entities = ast.literal_eval(entities)
        except Exception:
            return []

    labeled = classify_geo_entities_llm(entities)
    return [labeled.get(e, e) for e in entities]

In [18]:
df = pd.read_csv('../../data/events/3_countries/2000-2025.csv')
df = df.sample(n=100).reset_index(drop=True)

print("🔹 Нормализация местоположений")
df['norm_loc_v2'] = df['loc'].progress_apply(normalize_loc_list)

df.to_csv('../data/events/3_countries/2000-2025_norm.csv', index=False, encoding='utf-8')
print("✅ Готово! norm_loc_v2 добавлена.")
# работает нестабильно и долго. Отказываюсь от этого решения. Думаю в сторону regex + аппрув человека + показ нераспознанных сущностей в датасете. (Буду использовать spacy)

🔹 Нормализация loc...


Нормализация loc через LLM: 100%|██████████| 100/100 [04:52<00:00,  2.93s/it]

✅ Готово! norm_loc_v2 добавлена.
